# 05 · Model Evaluation

Evaluates all trained GNN checkpoints on the held-out test set, merges results with the classical ML baselines, and inspects confusion matrices / per-class performance.

**Prerequisite:** models trained (notebook 04 or `python main.py --stage train`) and baselines trained (`python main.py --stage baseline`).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from src import config


In [ ]:
from src.training.evaluate import evaluate_all_gnn_models, build_final_comparison

gnn_results = evaluate_all_gnn_models()
gnn_results

## Merge with baseline ML results and build the final comparison table

In [ ]:
comparison_df = build_final_comparison(gnn_results)
comparison_df

## Visualize the comparison

In [ ]:
import matplotlib.pyplot as plt

comparison_df.set_index("model_name")[["accuracy", "macro_f1"]].plot(
    kind="bar", figsize=(10, 5), title="Model comparison"
)
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

## Per-class performance breakdown for the best model

In [ ]:
from src.utils.metrics import get_per_class_metrics_df
from src.graph.dynamic_graph import IoTGraphDataset
from src.training.evaluate import load_trained_model, predict_on_test
from src.data.feature_selection import get_selected_features
from torch_geometric.loader import DataLoader

best_model_name = comparison_df.sort_values("macro_f1", ascending=False).iloc[0]["model_name"].lower()
print("Best model:", best_model_name)

if best_model_name in ("gcn", "graphsage", "gat"):
    test_dataset = IoTGraphDataset(split_name="test")
    test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False)
    model = load_trained_model(best_model_name, len(get_selected_features()))
    y_true, y_pred = predict_on_test(model, test_loader)
    display(get_per_class_metrics_df(y_true, y_pred))
else:
    print("Best model is a baseline; see results/metrics/ for its classification report.")

## Confusion matrices (saved figures)

In [ ]:
from IPython.display import Image, display

for model_name in ["gcn", "graphsage", "gat"]:
    path = os.path.join(config.FIGURES_DIR, f"confusion_matrix_{model_name}.png")
    if os.path.exists(path):
        print(model_name.upper())
        display(Image(filename=path))